# System 1 Notebook 00 - Import dữ liệu và chia batch

Notebook này dùng cho Kaggle/Colab. Chỉ cần cấu hình vài biến đường dẫn, rồi chạy lần lượt từ trên xuống dưới.

Luồng chính:
- clone/pull repo GitHub vào runtime
- cài `system1` trong VM hiện tại
- import dữ liệu từ link/folder ban tổ chức vào `AIC_DATA_ROOT`
- chạy `ingest` để tạo manifest release
- chạy `assign-batches` để chia batch cho worker

Lưu ý nhanh:
- Notebook này là notebook đầu tiên cần chạy trong System 1.
- Không cần tự chuẩn bị manifest CSV/JSON.
- Không cần tự tạo sẵn `raw_videos/` hoặc `metadata/`; importer sẽ tự đưa dữ liệu về đúng cấu trúc.
- Nên chạy tuần tự từ trên xuống dưới, tránh chạy nhảy cell khi chưa chắc trạng thái hiện tại.


## 0. Clone repo GitHub và cài System 1

Chạy section này đầu tiên để đưa code từ GitHub vào VM Kaggle/Colab và cài package ở chế độ editable.

Tùy chỉnh khi cần:
- Giữ nguyên `GITHUB_REPO_URL` nếu dùng repo chính thức.
- Set `AIC_REPO_ROOT` nếu repo đã được mount sẵn hoặc đã clone trước đó.
- Đổi `AIC_REPO_PARENT` nếu muốn clone repo sang thư mục khác trong VM.

Lưu ý tránh chạy sai:
- Nếu đổi branch/repo, code notebook và code trong `system1/` phải cùng version.
- Cell này có thể chạy `git pull`; nếu đã sửa code trực tiếp trong VM, hãy lưu lại trước.
- Cell này cài `gdown` để hỗ trợ import Google Drive folder.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import subprocess
import sys

GITHUB_REPO_URL = os.environ.get("GITHUB_REPO_URL", "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git")  # URL repo chứa code System 1
AIC_REPO_PARENT = Path(os.environ.get("AIC_REPO_PARENT", "/content" if Path("/content").exists() else "/kaggle/working" if Path("/kaggle/working").exists() else str(Path.cwd()))).expanduser().resolve()  # Thư mục cha để clone repo trong VM
AIC_REPO_ROOT_ENV = os.environ.get("AIC_REPO_ROOT")  # Nếu đã mount sẵn repo thì set biến này

if AIC_REPO_ROOT_ENV:
    REPO_ROOT = Path(AIC_REPO_ROOT_ENV).expanduser().resolve()
elif (Path.cwd() / "system1" / "pyproject.toml").exists():
    REPO_ROOT = Path.cwd().resolve()
else:
    REPO_ROOT = AIC_REPO_PARENT / "Multimodal-Agentic-Retrieval-Engine"
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        AIC_REPO_PARENT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", GITHUB_REPO_URL, str(REPO_ROOT)], check=True)

SYSTEM1_ROOT = REPO_ROOT / "system1"
subprocess.run([sys.executable, "-m", "pip", "install", "gdown", "-e", str(SYSTEM1_ROOT)], check=True)
os.environ["AIC_REPO_ROOT"] = str(REPO_ROOT)
print("REPO_ROOT=", REPO_ROOT)
print("SYSTEM1_ROOT=", SYSTEM1_ROOT)


## 1. Cấu hình đường dẫn và mode chạy

Đây là section duy nhất bạn cần chỉnh. Với Colab, hãy mount Drive trước nếu muốn lưu dữ liệu bền vững.

Tùy chỉnh khi cần:
- `AIC_ORGANIZER_SOURCE_URI`: folder/link dữ liệu từ ban tổ chức.
- `AIC_DATA_ROOT`: nơi notebook import dữ liệu thành `raw_videos/` và `metadata/`.
- `AIC_RUNTIME_ROOT`: nơi ghi output release, manifest và batch files.
- `AIC_ARTIFACT_ROOT`: thường giữ mặc định bằng `AIC_RUNTIME_ROOT`.
- `AIC_EXECUTION_MODE`: mode chạy pipeline, mặc định `gold_full`.
- `AIC_NUM_BATCHES`: số batch muốn chia cho teammate/worker chạy song song.

Lưu ý tránh chạy sai:
- Không trỏ `AIC_DATA_ROOT` và `AIC_RUNTIME_ROOT` vào cùng một thư mục nếu muốn dễ debug.
- Trên Colab, nên lưu vào Drive nếu cần giữ dữ liệu sau khi runtime tắt.
- Nếu đổi `AIC_NUM_BATCHES`, notebook worker phía sau phải dùng đúng batch mới tạo.
- Nếu chưa chắc mode nào phù hợp, giữ `gold_full` để dùng luồng chuẩn nhất.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

# from google.colab import drive
# drive.mount("/content/drive")

REPO_ROOT = Path(os.environ["AIC_REPO_ROOT"]).expanduser().resolve()
SYSTEM1_ROOT = REPO_ROOT / "system1"

AIC_ORGANIZER_SOURCE_URI = os.environ.get("AIC_ORGANIZER_SOURCE_URI", "")  # Folder nguồn từ ban tổ chức: path local/Drive hoặc link Google Drive folder
AIC_DATA_ROOT = Path(os.environ.get("AIC_DATA_ROOT", "/content/drive/MyDrive/aic/system1_data" if Path("/content/drive").exists() else "/kaggle/working/aic/system1_data" if Path("/kaggle/working").exists() else str(SYSTEM1_ROOT / "input"))).expanduser().resolve()  # Nơi lưu raw_videos/ và metadata/ sau khi import
AIC_RUNTIME_ROOT = Path(os.environ.get("AIC_RUNTIME_ROOT", "/content/drive/MyDrive/aic/system1_output" if Path("/content/drive").exists() else "/kaggle/working/aic/system1_output" if Path("/kaggle/working").exists() else str(SYSTEM1_ROOT / "output"))).expanduser().resolve()  # Nơi ghi release output
AIC_ARTIFACT_ROOT = Path(os.environ.get("AIC_ARTIFACT_ROOT", str(AIC_RUNTIME_ROOT))).expanduser().resolve()  # Nơi lưu artifact, mặc định dùng chung runtime root
AIC_EXECUTION_MODE = os.environ.get("AIC_EXECUTION_MODE", "debug_small_sample")  # Mode pipeline: debug_small_sample | bronze_fast | silver_balanced | gold_full
AIC_PROVIDER_MODE = os.environ.get("AIC_PROVIDER_MODE", "mock")  # Chế độ provider, thường để config để đọc configs/models.yaml
execution_mode = AIC_EXECUTION_MODE  # Alias ngắn để giữ notebook/test tương thích
provider_mode = AIC_PROVIDER_MODE  # Alias ngắn để giữ notebook/test tương thích
AIC_NUM_BATCHES = os.environ.get("AIC_NUM_BATCHES", "1")  # Số batch muốn chia ở notebook 00
worker_id = os.environ.get("AIC_WORKER_ID", "worker_000")  # Worker mặc định dùng ở các notebook worker
batch_id = os.environ.get("AIC_BATCH_ID", "batch_000")  # Batch mặc định để teammate chạy tiếp

release_dir = AIC_RUNTIME_ROOT / "competition_dataset_v001"
input_dir = AIC_DATA_ROOT
output_dir = AIC_RUNTIME_ROOT
num_batches = int(AIC_NUM_BATCHES)
RELEASE_DIR = release_dir
AIC_DATA_ROOT.mkdir(parents=True, exist_ok=True)
AIC_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
# Current CLI writes artifacts under output_dir / competition_dataset_v001 / artifacts. AIC_ARTIFACT_ROOT is reserved for future split-storage support.
AIC_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def run_cli(args: list[str]) -> None:
    cmd = [sys.executable, "-m", "system1.cli", *args]
    print("$", " ".join(str(part) for part in cmd))
    subprocess.run(cmd, cwd=SYSTEM1_ROOT, check=True)

def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

print("AIC_ORGANIZER_SOURCE_URI=", AIC_ORGANIZER_SOURCE_URI or "<chưa set>")
print("AIC_DATA_ROOT=", AIC_DATA_ROOT)
print("AIC_RUNTIME_ROOT=", AIC_RUNTIME_ROOT)
print("AIC_ARTIFACT_ROOT=", AIC_ARTIFACT_ROOT)
print("AIC_EXECUTION_MODE=", AIC_EXECUTION_MODE)
print("AIC_PROVIDER_MODE=", AIC_PROVIDER_MODE)
print("execution_mode=", execution_mode)
print("provider_mode=", provider_mode)
print("AIC_NUM_BATCHES=", AIC_NUM_BATCHES)
print("worker_id=", worker_id)
print("batch_id=", batch_id)

## 2. Import dữ liệu ban tổ chức vào `AIC_DATA_ROOT`

Section này tự động copy dữ liệu nguồn vào thư mục đích chuẩn của System 1. Mỗi lần chạy, importer sẽ làm sạch `raw_videos/` và `metadata/` cũ trong `AIC_DATA_ROOT` để tránh lẫn dữ liệu cũ.

Tùy chỉnh khi cần:
- Nếu ban tổ chức đưa Google Drive folder, dán link folder vào `AIC_ORGANIZER_SOURCE_URI`.
- Nếu dữ liệu đã nằm trong Drive/local runtime, set `AIC_ORGANIZER_SOURCE_URI` bằng path folder đó.
- Folder nguồn có thể có layout con khác nhau; importer sẽ tự dò video và metadata.

Lưu ý tránh chạy sai:
- Importer sẽ xóa lại `raw_videos/` và `metadata/` trong `AIC_DATA_ROOT`, nên đừng đặt `AIC_DATA_ROOT` vào thư mục chứa dữ liệu khác cần giữ.
- Nếu có 2 video trùng filename stem/video_id, importer sẽ báo lỗi để tránh conflict.
- Nếu thiếu metadata cho video, importer tạo metadata tối thiểu để pipeline vẫn chạy được.


In [ ]:
if AIC_ORGANIZER_SOURCE_URI:
    run_cli(["import-source", "--source-uri", AIC_ORGANIZER_SOURCE_URI, "--data-root", str(input_dir)])
else:
    print("Skip import-source because AIC_ORGANIZER_SOURCE_URI is empty.")
    print(f"Using existing input_dir={input_dir}")

## 3. Ingest và chia batch

Sau khi import xong, notebook chạy `ingest` để tạo manifest release và `assign-batches` để chia dữ liệu cho worker.

Tùy chỉnh khi cần:
- Đổi `AIC_EXECUTION_MODE` nếu muốn chạy nhẹ hơn để debug nhanh.
- Đổi `AIC_NUM_BATCHES` theo số teammate hoặc số phiên worker muốn chạy song song.

Lưu ý tránh chạy sai:
- Nếu đổi source data, chạy lại từ section import trước khi ingest.
- Nếu đổi số batch, mapping phân việc cho teammate cũng thay đổi.
- Notebook 00 chỉ chuẩn bị dữ liệu và chia batch; xử lý batch nằm ở notebook worker sau.


In [ ]:
run_cli(["ingest", "--mode", execution_mode, "--input", str(input_dir), "--output", str(output_dir)])
run_cli(["assign-batches", "--mode", execution_mode, "--num-batches", str(num_batches), "--output", str(output_dir)])

## 4. Kiểm tra nhanh kết quả

Section này in ra report import, dataset manifest và nội dung batch đầu tiên để bạn xác nhận notebook chạy đúng.

Bạn nên kiểm tra nhanh:
- `organizer_import_report.json` có đúng source và số file import không.
- `dataset_manifest.json` đã được sinh ra sau bước ingest không.
- `batch_000.txt` có danh sách `video_id` để giao cho worker không.

Lưu ý tránh chạy sai:
- Nếu import report đúng nhưng manifest rỗng, kiểm tra lại source folder và `AIC_DATA_ROOT`.
- Nếu batch chưa đúng ý, chỉnh `AIC_NUM_BATCHES` rồi chạy lại từ section ingest/chia batch.


In [ ]:
import csv
import pandas as pd
from IPython.display import JSON, display

import_report_path = input_dir / "organizer_import_report.json"
if import_report_path.exists():
    print("organizer_import_report=")
    display(JSON(load_json(import_report_path)))

videos_path = release_dir / "tables" / "videos.parquet"
videos = pd.read_parquet(videos_path)
print(f"videos_count={len(videos)}")
display(videos[["video_id", "video_ref"]])

for report_name in ["dataset_report.json", "dataset_manifest.json"]:
    report_path = release_dir / "manifests" / report_name
    if report_path.exists():
        print(f"{report_name}=")
        display(JSON(load_json(report_path)))

batch_manifest_path = release_dir / "manifests" / "batch_manifest.csv"
if batch_manifest_path.exists():
    with batch_manifest_path.open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    print(f"batch_manifest_rows={len(rows)}")
    display(pd.DataFrame(rows))

print("generated batch files:")
for path in sorted((release_dir / "manifests").glob("batch_*.txt")):
    print(f"--- {path.name} ---")
    print(path.read_text(encoding="utf-8").strip())